# 第1回：LLMを動かして理解する

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session01/session01_llm_basics.ipynb)

このノートブックでは LLMの動作を手を動かしながら体験する。


## 準備
---

> **GPU ランタイムに切り替えて使用する（T4 以上推奨）。**  
> Colab メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU
> GPUランタイムが利用できない場合はCPUでも可（ただし遅い）

### パッケージのインストール

In [ ]:
# パッケージのインストール
!pip install -q transformers accelerate openai
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!uv pip install "vllm==0.19.1" --torch-backend=cu129 -q
#!uv pip install --system "vllm==0.19.1" -q


### `OPENAI_API_KEY` の設定

OpenAI API キー（`sk-...`）を [OpenAI Platform](https://platform.openai.com/api-keys) で取得し、次の方法で設定


1. Colab 左サイドバーの **鍵アイコン（シークレット）** を開く
2. **「新しいシークレットを追加」** をクリック
3. 名前に `OPENAI_API_KEY`、値に `sk-...` を入力して保存
4. **「ノートブックアクセス」** をオンにする
5. 以下のセルを実行


In [16]:
# OPENAI_API_KEYの設定
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

---
# LLM の基本

## LLM とは何か

LLM（Large Language Model / 大規模言語モデル）は一言で言えば **「テキストを受け取って次の単語（正確にはトークン（後述））を1つ予測して返す関数」**

[![](https://mermaid.ink/img/pako:eNpdjc1ugzAQhF_F2lMrkUCMzY-VRq2aY3rrqXUVOeAAKtjIGCUt4t2LadNI2cvujma-GSDTuQQGx1qfslIYi163XKFpKtX2dm_l2b77a-faXJW1Pwv-B1osNmi3e5m37u1tRE33_qRN_p_4haOnxwF1pWglQwcjMtl5qBYHWTPE4SaKHlzD3bX9_g_GAY3gQWGqHJg1vfSgkaYR7oXBFXGwpWwkB4fNhfnkwJXLtEK9ad1cYkb3RXl5-jYXVm4rURgxOY6i7pxFqlyaZ90rC4xiMjOADXAGtsJkSWlM4ojGNAmTOPLga5LDdElohAnGCSFBkpLRg--5NljGYTpRVjgKKE5oRMYfgqx8ew?type=png)](https://mermaid.live/edit#pako:eNpdjc1ugzAQhF_F2lMrkUCMzY-VRq2aY3rrqXUVOeAAKtjIGCUt4t2LadNI2cvujma-GSDTuQQGx1qfslIYi163XKFpKtX2dm_l2b77a-faXJW1Pwv-B1osNmi3e5m37u1tRE33_qRN_p_4haOnxwF1pWglQwcjMtl5qBYHWTPE4SaKHlzD3bX9_g_GAY3gQWGqHJg1vfSgkaYR7oXBFXGwpWwkB4fNhfnkwJXLtEK9ad1cYkb3RXl5-jYXVm4rURgxOY6i7pxFqlyaZ90rC4xiMjOADXAGtsJkSWlM4ojGNAmTOPLga5LDdElohAnGCSFBkpLRg--5NljGYTpRVjgKKE5oRMYfgqx8ew)

ただし普通の関数とは異なり、内部に **膨大な数のパラメータ（重み）** を持つ。  
期待する出力に近づくように重みを少しずつ調整することを**学習**という。

予測した単語を、入力テキストの末尾に足すという処理の繰り返す（**生成ループ**（自己回帰生成））ことで文章を生成することができる：

[![](https://mermaid.ink/img/pako:eNpdjstuwjAQRX_Fmm1DHo6dh0XZlCXsuiqpkIVNEpXYkWsLaJR_r5MKoWY2M3N17p0Z4KSFBAbni76eGm4set9WCvlqVe_s0cqbPUTrido8lXU0C9EnWq02aLfbz107u7QoPx-v2oj_Dn5YJqKXV7SE_9gnAwHUphXArHEygE6ajk8rDNPHFdhGdrIC5kfBzVcFlRq9p-fqQ-vuYTPa1c1jcb3gVm5bXhvuiTO_fE-IVEKaN-2UBYZxMmcAG-AGLMEkpDQneUZzWqRFngVw93JahoRmmGBcEBIXJRkD-JnPxmGelhSTBGcxxQXNyPgLbWd25w?type=png)](https://mermaid.live/edit#pako:eNpdjstuwjAQRX_Fmm1DHo6dh0XZlCXsuiqpkIVNEpXYkWsLaJR_r5MKoWY2M3N17p0Z4KSFBAbni76eGm4set9WCvlqVe_s0cqbPUTrido8lXU0C9EnWq02aLfbz107u7QoPx-v2oj_Dn5YJqKXV7SE_9gnAwHUphXArHEygE6ajk8rDNPHFdhGdrIC5kfBzVcFlRq9p-fqQ-vuYTPa1c1jcb3gVm5bXhvuiTO_fE-IVEKaN-2UBYZxMmcAG-AGLMEkpDQneUZzWqRFngVw93JahoRmmGBcEBIXJRkD-JnPxmGelhSTBGcxxQXNyPgLbWd25w)



### LLMの学習ステップ

#### STEP1 事前学習
大量のテキスト（Web、書籍、コードなど）から、**「次の単語予測」をひたすら練習させる**段階が **事前学習** だ。「テキストの続きを当てるクイズ」を膨大な量こなすイメージで、例えば「日本の首都は」の次に「東京」が来やすい、という統計的パターンを重みに刻み込んでいく。ここで得られるのが **ベースモデル** で、言語の統計的パターンは身につくが、質問応答や会話の「指示に従う」能力はそのままだと弱い。

例（次トークン予測のイメージ）:
- 入力: 「日本の首都は」
- 期待: 「東京」

#### STEP2 教師ありファインチューニング（インストラクション・チューニング）
人間（または人手で整形したデータ）が用意した **「指示 → 望ましい回答」** のペアで追加学習し、ユーザーの意図に沿った出力（指示追従）を身につけさせる。一般に **SFT（Supervised Fine-Tuning）** と呼ばれる。

1. Alpaca形式（非チャット）
    「1つのプロンプト文字列に指示と入力を詰めて、1つの出力を学習する」形式。

    ```
    Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

    ### Instruction:
    次の文を英語に翻訳してください。

    ### Input:
    こんにちは

    ### Response:
    ```

2. ChatML形式（チャット）
    system/user/assistant の **役割（role）付きメッセージ列** を入力として学習し、会話として自然に応答する形を教える。推論時も同様のメッセージ形式を使う。複数ターンの指示応答が可能。


    ```
    <|im_start|>system
    簡潔に答えてください。<|im_end|>
    <|im_start|>user
    東京について教えてください。<|im_end|>
    <|im_start|>assistant
    ```

    > `<|im_start|>`/`<|im_end|>`はroleとmessageの区切りを表す特殊トークン

#### STEP3 RLHF
SFT だけだと「それっぽいが微妙に不正確」「安全でない」「冗長」などが残りやすい。そこで、人間の好み（どちらが良い回答か）を使って **より望ましい応答** に寄せる追加学習を行う。
代表例が **RLHF（Reinforcement Learning from Human Feedback）** で、概念的には次の流れ：
- 同じ質問に対する複数の回答候補を作る
- 人間（またはポリシー）で順位付けする
- その「好ましさ」を学習し、モデルの出力を改善する

---
## ⚠️ 重要：ChatGPT は LLM ではない

> **チャットアプリ（サービス）と LLM（モデル）は別物**

| サービス（アプリ） | LLM（モデル） | 補足 |
|------------------|---------------|------|
| **ChatGPT**（OpenAI） | **GPT-4o** など | 名前は別。「GPT」はモデル系列の総称 |
| **Claude**（Anthropic） | **Claude Haiku / Sonnet / Opus** | **アプリ名とモデル名がどちらも「Claude」で混同しやすい** |
| **Gemini**（Google） | **Gemini 2.5 Pro / Flash** など | **アプリ名とモデル名がどちらも「Gemini」で混同しやすい** |


「前の会話を覚えている」のは **アプリ側の会話履歴** 。LLM 本体は **ステートレス**（呼び出しのたびに記憶を持たない）なので、アプリが毎回「履歴＋新しいメッセージ」をまとめて渡している。

[![](https://mermaid.ink/img/pako:eNp1U8Fu2zAM_RVCpw5LM9uJ1VgYAgwdsEsz9LBeBl8Ui3WE2ZImyUWyIP8-yk6ydcV8sSjyPT4-SUfWWIVMsIA_BzQNftay9bKvDdAnm2g9PAX0IMP4n_ad9FE32kkT4ZNzKXm_kxG2Nt6kxZfHb--myq3dw-PgXYewoUYd-MFE3eOU_ZdrKiG2h4fNzR8SNKo203LUcrtep7YCAmWgxxBke2ZMaq7p7aA7BT5NFuLHrf-wDocQsQdtQvRDE7U1Ad5Dk7TvdKBhDxQOqceV9TXvqFBAiwa9jAiN7Wm2RDTVfbW0aV-I4FxpDVK7Z_TJXGhk14EOECKBO2oxqmqsibiPKeG8fdEKFWEg7vAi_iJjMuj2OqEMgWQn6_7vgnQu2fSmFKJ9Nflf0IRNTgtQOrhOHt6i2Yy1XismyEicsR59L1PIjomnZiSeTpkJWirpf9SsNifC0DF_t7a_wLwd2t0lGJwiV84XkIln2YVUQuLR31u6N0wU5WI5kjBxZHsmcp7Ns4LzvKw451W54jN2YKK8m1fZYnmXL3Ne5it-mrFfY9dsvirLbFUsqoJXRcnLasZQaRp_Mz2D8TWcfgNkWAIV?type=png)](https://mermaid.live/edit#pako:eNp1U8Fu2zAM_RVCpw5LM9uJ1VgYAgwdsEsz9LBeBl8Ui3WE2ZImyUWyIP8-yk6ydcV8sSjyPT4-SUfWWIVMsIA_BzQNftay9bKvDdAnm2g9PAX0IMP4n_ad9FE32kkT4ZNzKXm_kxG2Nt6kxZfHb--myq3dw-PgXYewoUYd-MFE3eOU_ZdrKiG2h4fNzR8SNKo203LUcrtep7YCAmWgxxBke2ZMaq7p7aA7BT5NFuLHrf-wDocQsQdtQvRDE7U1Ad5Dk7TvdKBhDxQOqceV9TXvqFBAiwa9jAiN7Wm2RDTVfbW0aV-I4FxpDVK7Z_TJXGhk14EOECKBO2oxqmqsibiPKeG8fdEKFWEg7vAi_iJjMuj2OqEMgWQn6_7vgnQu2fSmFKJ9Nflf0IRNTgtQOrhOHt6i2Yy1XismyEicsR59L1PIjomnZiSeTpkJWirpf9SsNifC0DF_t7a_wLwd2t0lGJwiV84XkIln2YVUQuLR31u6N0wU5WI5kjBxZHsmcp7Ns4LzvKw451W54jN2YKK8m1fZYnmXL3Ne5it-mrFfY9dsvirLbFUsqoJXRcnLasZQaRp_Mz2D8TWcfgNkWAIV)

---
## オープンモデルとクローズドモデル

モデルは重みの公開状況によって2種類に分かれる。

### クローズドモデル（重みは非公開）

プロバイダが重みを非公開にし、**API 経由でのみ利用できる**モデル。

| プロバイダ | モデル例 |
|-----------|----------|
| Anthropic | Claude Sonnet 4.6, Claude Opus 4.7 |
| OpenAI | GPT-4o, o3 |
| Google | Gemini 2.5 Pro, Gemini 2.5 Flash |

---

### オープンモデル（重みは公開）

モデルの重みが公開されており、**ダウンロードして手元で動かせる**モデル。

| 開発元 | モデル例 |
|--------|----------|
| Meta | Llama 3.3, Llama 4 |
| Mistral AI | Mistral 7B, Mixtral 8x7B |
| Google | Gemma 3 |
| Alibaba | Qwen 3 |
| Microsoft | Phi-4 |

モデルを動かす（推論）実行基盤（ランタイム）

| ランタイム | 対応形式 | 特徴 |
|-----------|---------|------|
| [llama.cpp](https://github.com/ggml-org/llama.cpp) | GGUF | CPU でも動く軽量推論エンジン（C++）。多くのツールの**中核バックエンド** |
| [llama-cpp-python](https://github.com/abetlen/llama-cpp-python) | GGUF | **llama.cpp** の Python バインディング。スクリプトから直接推論 |
| [Ollama](https://ollama.com) | GGUF | ローカル向け CLI。バックエンドに **llama.cpp** を使用 |
| [LM Studio](https://lmstudio.ai) | GGUF / MLX | デスクトップ向け GUI。バックエンドに **llama.cpp**（ほか GGUF 対応エンジン）を使用 |
| [vLLM](https://github.com/vllm-project/vllm) | Safetensors（HF） / AWQ / GPTQ / FP8 | GPU サーバー向け高速推論フレームワーク |
| [SGLang](https://github.com/sgl-project/sglang) | Safetensors（HF） / AWQ / GPTQ / FP8 | GPU サーバー向け。高速推論・構造化生成に強い |
| [Transformers](https://huggingface.co/docs/transformers) | Safetensors（HF） / PyTorch (.bin) | 学習・ファインチューニング向け Python ライブラリ（PyTorch 上で推論も可能） |



> **Transformers** は推論ランタイムではなくディープラーニングフレームワーク上で動く学習ライブラリなので、用途レイヤーが異なる。今回は推論ランタイムでは隠ぺいされているトークン化や生成ループなどの内部処理を直接見るために使う。

[![](https://mermaid.ink/img/pako:eNpVkbFugzAURX_FejOghEAoHrqkUjsQZUinlg6v8AJWwabGpEmj_HuNCVLqwbLvOdeW7AsUqiTg8D1gqVGaTY3a5JLZYYRpiO06kqy1UsOEPJAmWRDTgzSipX4STz6eRM8yVWDDfP-R7UkfSU_wPMMfhp-90VgYoaTTXkRV34e5nCrZlu3NUArF2fsiWHpsEaQfE9o1DbY45fEIHuIbccAvus7vzqZW0jnhqKxX90pglYm5fpzc4DHLti5PXL6az90_ZyirOxLdwKt9rv6gdEu6d9jR5dybZvCg0qIEbvRAHli3xXELl5HnYGpqKQdulyXqrxxyebWdDuWbUu1c02qo6nkzdCUaehJYabTGAZt-VEiWpDfKfgzw5CF1ZwC_wAn4arEO4nUYJUmUhskyjNcenIFH6f80vXrw6261z5rE1z95CKCf?type=png)](https://mermaid.live/edit#pako:eNpVkbFugzAURX_FejOghEAoHrqkUjsQZUinlg6v8AJWwabGpEmj_HuNCVLqwbLvOdeW7AsUqiTg8D1gqVGaTY3a5JLZYYRpiO06kqy1UsOEPJAmWRDTgzSipX4STz6eRM8yVWDDfP-R7UkfSU_wPMMfhp-90VgYoaTTXkRV34e5nCrZlu3NUArF2fsiWHpsEaQfE9o1DbY45fEIHuIbccAvus7vzqZW0jnhqKxX90pglYm5fpzc4DHLti5PXL6az90_ZyirOxLdwKt9rv6gdEu6d9jR5dybZvCg0qIEbvRAHli3xXELl5HnYGpqKQdulyXqrxxyebWdDuWbUu1c02qo6nkzdCUaehJYabTGAZt-VEiWpDfKfgzw5CF1ZwC_wAn4arEO4nUYJUmUhskyjNcenIFH6f80vXrw6261z5rE1z95CKCf)



---
# Hands-on

以下の5セクションで**ベースモデル**と**チャットモデル**を動かし比較します。

| セクション | ランタイム | 種別 |
|-----------|----------|------|
| A | Transformers | オープンモデル（Python ライブラリ） |
| ~~B~~ | ~~vLLM~~ | ~~オープンモデル（GPU 高速推論）~~ |
| ~~C~~ | ~~Ollama~~ | ~~オープンモデル（CLI / REST API）~~ |
| ~~D~~ | ~~OpenAI~~ | ~~クローズドモデル（API）~~ |




- **ベースモデル** — 事前学習のみ。テキストの「続き」を生成する
- **チャットモデル（Instruct）** — 指示チューニング済み。質問・指示に適切に応答する

---
## A. Transformers（オープンモデル）

### A-0. TokenとTokenizer

**token**とは、モデルが扱うテキストの意味的な最小単位（自然言語の単語相当）。1文字や1単語に必ずしも対応せず、モデルごとに決まった語彙（vocabulary）の中の1要素として数えられる。同じ見た目の文字列でも、言語や文脈によって分割の仕方が変わる。tokenの実体はtoken IDと呼ばれる整数。テキストとtoken ID列を相互に変換する変換器を**tokenizer**という。

モデルが受け取るのは文字列そのものではなく、**token ID列**。




In [ ]:
from huggingface_hub import login

# モデルとトークナイザーをダウンロードするためにHugging Faceにログイン
login()
print('Hugging Face ログイン完了')

In [10]:
from transformers import AutoTokenizer

model_id = 'Qwen/Qwen2.5-0.5B'
base_tokenizer = AutoTokenizer.from_pretrained(model_id)

samples = [
    '東京は日本の首都であり、',
    '東京',
    'Tokyo',
    'AI',
]

print('[Transformers / トークナイザー]')
for text in samples:
    ids = base_tokenizer.encode(text)
    tokens = base_tokenizer.convert_ids_to_tokens(ids)
    print(f'入力: {text!r}')
    print(f'  token IDs: {ids}')
    print(f'  tokens:    {tokens}')
    print(f'  decode:    {base_tokenizer.decode(ids)!r}')
    print()

### 推論ランタイムの内部処理

[![](https://mermaid.ink/img/pako:eNqNVE1vozAQ_SuWT6mWfBDzEVC3W2m7hz1UPWwvu6WqHJgkqGAjx-y2Jfnva-xACAlSc4js9968sWcGVzjmCeAQrzL-L95QIdHjXcSQ-qWsKOVthbYbWkCI4lTEGVgoo0vIQhThnzUfYbQ3cvO_LZdrQYsNEiWTaQ4Gbe1eCsHzQj5Nr-u0N2Z3PdWb6fNRnCsgG61ouKJjwZdcovsauToqJH8Fln6AeAFWh48ibOQZZeuSrgE9Nopj0MhoryJ80SmBzzsZ7amTuaL22zZX7GIXLqqJRsvg7SC9oIw5S1KZclb9ePhl4r7tT-iYyqfzpOjLV3Rm3PHlpTw_9QnYOwywxCyM6DghyTIb11PSmZEHLekPScQ6E4bAvh2Pb07mo0MfEARzreq33Sj7KALS8TS36HoeKgOOVulhM7ReInCPuToJEHgabzthuHaLwFf8eMf47tCOVqDWCBYDZ-oYBNrgHba70750C96e3p71SmJmsl8Sgyq5qbMx6bZDNaCKS_FXtTBLGVBxaJYq-WWcDOBOi-eccckZ_G4odyDEG8D9YavFMBUMU6pYlzPZA7fHFl6LNMGhFCVYOAeR03qLq5qPsNyAet5wPeUJFa8RjthexRSU_eE8b8IEL9ebZlMWCZVwl1L1RCrFimbbWqI-KBDfuXowcUiItsBhhd9wGDgTMncC4gW-7xFvYVv4HYf2bDEJfMe158QlMzIPnL2FP3TS2SQIvMB1FoQEtu-6noVBzRYX9-aV14_9_j-ETfJ8?type=png)](https://mermaid.live/edit#pako:eNqNVE1vozAQ_SuWT6mWfBDzEVC3W2m7hz1UPWwvu6WqHJgkqGAjx-y2Jfnva-xACAlSc4js9968sWcGVzjmCeAQrzL-L95QIdHjXcSQ-qWsKOVthbYbWkCI4lTEGVgoo0vIQhThnzUfYbQ3cvO_LZdrQYsNEiWTaQ4Gbe1eCsHzQj5Nr-u0N2Z3PdWb6fNRnCsgG61ouKJjwZdcovsauToqJH8Fln6AeAFWh48ibOQZZeuSrgE9Nopj0MhoryJ80SmBzzsZ7amTuaL22zZX7GIXLqqJRsvg7SC9oIw5S1KZclb9ePhl4r7tT-iYyqfzpOjLV3Rm3PHlpTw_9QnYOwywxCyM6DghyTIb11PSmZEHLekPScQ6E4bAvh2Pb07mo0MfEARzreq33Sj7KALS8TS36HoeKgOOVulhM7ReInCPuToJEHgabzthuHaLwFf8eMf47tCOVqDWCBYDZ-oYBNrgHba70750C96e3p71SmJmsl8Sgyq5qbMx6bZDNaCKS_FXtTBLGVBxaJYq-WWcDOBOi-eccckZ_G4odyDEG8D9YavFMBUMU6pYlzPZA7fHFl6LNMGhFCVYOAeR03qLq5qPsNyAet5wPeUJFa8RjthexRSU_eE8b8IEL9ebZlMWCZVwl1L1RCrFimbbWqI-KBDfuXowcUiItsBhhd9wGDgTMncC4gW-7xFvYVv4HYf2bDEJfMe158QlMzIPnL2FP3TS2SQIvMB1FoQEtu-6noVBzRYX9-aV14_9_j-ETfJ8)


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def generate_loop(model, tokenizer, input_tokens, max_new_tokens=60, verbose_steps=8):
    """自己回帰の生成ループ：推論 → 次トークン選択 → 入力に連結 → 繰り返し"""
    eos_id = tokenizer.eos_token_id
    new_token_ids = []
    input_tokens = input_tokens.clone()

    for step in range(max_new_tokens):
        with torch.no_grad():
            # logits: 語彙の各トークンに対する「次トークンらしさ」の生スコア
            # shape は (batch, sequence_length, vocab_size)。[:, -1, :] で末尾位置のみを取り出す
            # ※logitsにはinput_tokens末尾の次トークンだけでなく、途中の位置の次トークンスコアも入っている（学習時に使用）
            logits = model(input_tokens).logits[:, -1, :]
        next_token = logits.argmax(dim=-1).unsqueeze(-1)  # logitsから次トークンの選び方は色々あるが、ここでは最大スコア（最大確率）のトークンを選ぶ(greedy decoding)
        next_token_id = next_token.item()
        new_token_ids.append(next_token_id)

        if step < verbose_steps:
            print(f"  step {step + 1}: next_token_id={next_token_id} → {tokenizer.decode([next_token_id])!r}")

        input_tokens = torch.cat([input_tokens, next_token], dim=1)

        if eos_id is not None and next_token_id == eos_id:
            if step < verbose_steps:
                print("  （EOS で停止）")
            break
    else:
        if verbose_steps > 0 and max_new_tokens > verbose_steps:
            print(f"  ...（以降 {max_new_tokens - verbose_steps} トークンは省略表示）")

    return input_tokens, new_token_ids

**Qwen2.5-0.5B**(ベースモデル)で生成

In [12]:
base_model_id = 'Qwen/Qwen2.5-0.5B'
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

prompt = '東京は日本の首都であり、'
inputs = base_tokenizer(prompt, return_tensors='pt').to(base_model.device)

print('[Transformers / ベースモデル / 生成ループ（先頭数ステップ）]')
print('入力:', repr(prompt))
print()
_, new_ids = generate_loop(
    base_model, base_tokenizer, inputs['input_ids'], max_new_tokens=60, verbose_steps=8
)
print()
print('[生成結果（追加分のみ）]')
print(base_tokenizer.decode(new_ids, skip_special_tokens=True))

**Qwen2.5-0.5B-Instruct**（チャットモデル）で生成

> 入力は **メッセージ列（役割付き）** を `apply_chat_template()` でチャットテンプレート文字列に変換してから渡す

In [13]:
chat_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

messages = [
    {'role': 'system', 'content': '簡潔に答えてください。'},
    {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
]

# apply_chat_template でメッセージ列をチャットテンプレート文字列に変換
# add_generation_prompt=True で末尾に「assistantの発話開始」マーカーを付与し、続きを生成させる
prompt_text = chat_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print('[apply_chat_template の出力（モデルに渡す実際の文字列）]')
print(prompt_text)
print()

inputs = chat_tokenizer(prompt_text, return_tensors='pt').to(chat_model.device)

print('[Transformers / チャットモデル / 生成ループ（先頭数ステップ）]')
_, new_ids = generate_loop(
    chat_model, chat_tokenizer, inputs['input_ids'], max_new_tokens=200, verbose_steps=8
)
print()
print('[生成結果（追加分のみ）]')
print(chat_tokenizer.decode(new_ids, skip_special_tokens=True))


GPUメモリを解放するために変数の参照を削除してPyTorchのキャッシュをクリア

In [ ]:
import gc
import torch

for name in [
    "base_tokenizer",
    "base_model",
    "chat_tokenizer",
    "chat_model",
    "tokenizer",
    "model"
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()